# Rubin Schedule (opsim) First-Visit-of-Night Analysis

Explore `baseline_v5.3.0_10yrs.db`: pull the first visit of every simulated night, then summarize starting azimuth, altitude, rotator angle, and filter — broken down by survey (WFD, DDF, twilight, templates), with ToO nights excluded, and the current initial telescope alignment target (Az=0, El=70, Rot=0, filter=i) overlaid for reference.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

DB_PATH = "baseline_v5.3.0_10yrs.db"
CSV_PATH = "first_visit_per_night.csv"

conn = sqlite3.connect(DB_PATH)

query = """
SELECT o.night, o.observationStartMJD,
       o.azimuth, o.altitude,
       o.rotTelPos, o.rotSkyPos,
       o.filter, o.band, o.scheduler_note
FROM observations o
JOIN (
    SELECT night, MIN(observationStartMJD) AS start_mjd
    FROM observations
    GROUP BY night
) f ON o.night = f.night AND o.observationStartMJD = f.start_mjd
ORDER BY o.night;
"""

first_visits = pd.read_sql_query(query, conn)
conn.close()

def survey_label(note):
    token = note.split(',')[0].strip()
    if token.startswith('DD'):
        return 'DDF'
    if token.startswith('greedy'):
        return 'WFD (greedy)'
    if token.startswith('pair'):
        return 'WFD (pairs)'
    if token.startswith('templates'):
        return 'templates'
    if token.startswith('twilight'):
        return 'twilight'
    if token == 'ToO':
        return 'ToO'
    if token.startswith('blob'):
        return 'WFD (blob)'
    return token

first_visits["survey"] = first_visits["scheduler_note"].apply(survey_label)
first_visits.to_csv(CSV_PATH, index=False)
print(f"{len(first_visits)} nights written to {CSV_PATH}")


## 1. Load the first-visit-per-night data and exclude ToO nights

In [ ]:
first_visits = pd.read_csv("first_visit_per_night.csv")
first_visits = first_visits[first_visits["survey"] != "ToO"].reset_index(drop=True)
n_nights = len(first_visits)
print(f"{n_nights} nights remain after excluding ToO")
first_visits.head()


## 2. Color schemes, style helper, and the current initial-alignment target

In [ ]:
# Rubin/RTN-045 colorblind-friendly ugrizy filter colors
filter_colors = {
    'u': '#1600ea', 'g': '#31de1f', 'r': '#b52626',
    'i': '#370201', 'z': '#ba52ff', 'y': '#61a2b3',
}
band_order = ['u', 'g', 'r', 'i', 'z', 'y']

survey_colors = {
    'WFD (greedy)': '#4c72b0',
    'WFD (pairs)':  '#64b5cd',
    'WFD (blob)':   '#8172b2',
    'DDF':          '#c44e52',
    'twilight':     '#dd8452',
    'templates':    '#937860',
}
survey_order = list(survey_colors.keys())

# Current initial telescope alignment strategy
ALIGNMENT = {"az": 150, "el": 70, "rot": 0, "filter": "i"}
ALIGNMENT_COLOR = "black"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#444444",
    "axes.grid": True,
    "axes.axisbelow": True,   # gridlines drawn behind data/plot objects
    "grid.color": "#dddddd",
    "grid.linewidth": 0.6,
    "font.size": 11,
})

def style(ax):
    ax.set_axisbelow(True)  # belt-and-suspenders: force gridlines behind bars/points/lines
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)


## 3. Summary statistics

In [ ]:
summary_cols = ["azimuth", "altitude", "rotTelPos", "rotSkyPos"]
first_visits[summary_cols].describe()


In [ ]:
first_visits["band"].value_counts()


## 4. Nights starting in each filter, with the alignment filter (i-band) marked

In [ ]:
band_counts = first_visits["band"].value_counts().reindex(band_order).fillna(0)

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(band_counts.index, band_counts.values,
              color=[filter_colors[b] for b in band_counts.index],
              edgecolor="black", linewidth=0.6, zorder=3)
for bar, cnt in zip(bars, band_counts.values):
    ax.annotate(f"{int(cnt)}", (bar.get_x() + bar.get_width()/2, bar.get_height()),
                textcoords="offset points", xytext=(0, 4), ha="center", fontsize=10)

# mark the current initial-alignment filter
align_idx = band_order.index(ALIGNMENT["filter"])
align_bar = bars[align_idx]
ax.annotate("Current IA",
            xy=(align_bar.get_x() + align_bar.get_width()/2, align_bar.get_height()*1.5),
            xytext=(0, 48), textcoords="offset points", ha="center", fontsize=9,
            color=ALIGNMENT_COLOR,
            arrowprops=dict(arrowstyle="->", color=ALIGNMENT_COLOR))

ax.set_ylabel("Number of nights")
ax.set_xlabel("Filter (band)")
ax.set_title(f"Nights starting in each filter, excl. ToO (n={n_nights})")
style(ax)
plt.tight_layout()
plt.show()


## 5. Azimuth / altitude / rotator histograms, with alignment target marked

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(first_visits["azimuth"], bins=36, color="#3b6fa0", edgecolor="black", linewidth=0.5, zorder=3)
axes[0].axvline(ALIGNMENT["az"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
                label=f"initial alignment (Az={ALIGNMENT['az']})")
axes[0].set_title("Starting Azimuth"); axes[0].set_xlabel("azimuth (deg)"); axes[0].set_ylabel("Number of nights")
axes[0].legend(fontsize=8, frameon=False)

axes[1].hist(first_visits["altitude"], bins=36, color="#3b6fa0", edgecolor="black", linewidth=0.5, zorder=3)
axes[1].axvline(ALIGNMENT["el"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
                label=f"initial alignment (El={ALIGNMENT['el']})")
axes[1].set_title("Starting Altitude"); axes[1].set_xlabel("altitude (deg)")
axes[1].legend(fontsize=8, frameon=False)

axes[2].hist(first_visits["rotTelPos"], bins=36, color="#3b6fa0", edgecolor="black", linewidth=0.5, zorder=3)
axes[2].axvline(ALIGNMENT["rot"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
                label=f"initial alignment (Rot={ALIGNMENT['rot']})")
axes[2].set_title("Starting Rotator Angle (rotTelPos)"); axes[2].set_xlabel("rotTelPos (deg)")
axes[2].legend(fontsize=8, frameon=False)

for ax in axes:
    style(ax)
plt.suptitle(f"Distributions of first-visit-of-night pointing parameters, excl. ToO (n={n_nights})", y=1.03)
plt.tight_layout()
plt.show()


## 6. Starting azimuth composition by filter, stacked

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(0, 360, 37)
bottom = np.zeros(len(bins) - 1)
for b in band_order:
    subset = first_visits.loc[first_visits["band"] == b, "azimuth"]
    counts, _ = np.histogram(subset, bins=bins)
    ax.bar(bins[:-1], counts, width=np.diff(bins), align="edge", bottom=bottom,
           color=filter_colors[b], edgecolor="black", linewidth=0.3, label=b, zorder=3)
    bottom += counts

ax.axvline(ALIGNMENT["az"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
           label=f"initial alignment (Az={ALIGNMENT['az']})")

ax.set_xlabel("azimuth (deg)"); ax.set_ylabel("Number of nights")
ax.set_title("Starting azimuth composition by filter, excl. ToO")
ax.legend(title="band", ncol=7, loc="upper center", bbox_to_anchor=(0.5, -0.15), frameon=False, fontsize=8)
style(ax)
plt.tight_layout()
plt.show()


## 7. Starting azimuth vs. altitude, colored by survey, with alignment target marked

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for s in survey_order:
    sub = first_visits[first_visits["survey"] == s]
    if len(sub) == 0:
        continue
    ax.scatter(sub["azimuth"], sub["altitude"], s=18, alpha=0.7, color=survey_colors[s],
               edgecolor="none", label=f"{s} (n={len(sub)})", zorder=3)

ax.scatter([ALIGNMENT["az"]], [ALIGNMENT["el"]], marker="*", s=350, color="gold",
           edgecolor="black", linewidth=1.4, zorder=5,
           label=f"initial alignment (Az={ALIGNMENT['az']}, El={ALIGNMENT['el']})")

ax.set_xlabel("azimuth (deg)"); ax.set_ylabel("altitude (deg)"); ax.set_xlim(0, 360)
ax.set_title(f"Starting azimuth vs. altitude by survey, excl. ToO (n={n_nights})")
ax.legend(title="survey", loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, fontsize=9)
style(ax)
plt.tight_layout()
plt.show()


## 8. Azimuth and altitude, stacked by survey, with alignment target marked

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

az_bins = np.linspace(0, 360, 37)
bottom = np.zeros(len(az_bins) - 1)
for s in survey_order:
    subset = first_visits.loc[first_visits["survey"] == s, "azimuth"]
    counts, _ = np.histogram(subset, bins=az_bins)
    axes[0].bar(az_bins[:-1], counts, width=np.diff(az_bins), align="edge", bottom=bottom,
                color=survey_colors[s], edgecolor="black", linewidth=0.3, label=s, zorder=3)
    bottom += counts
axes[0].axvline(ALIGNMENT["az"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
                label=f"initial alignment (Az={ALIGNMENT['az']})")
axes[0].set_xlabel("azimuth (deg)"); axes[0].set_ylabel("Number of nights")
axes[0].set_title("Starting azimuth by survey, excl. ToO")
axes[0].legend(fontsize=7, frameon=False, loc="upper right")

alt_bins = np.linspace(first_visits["altitude"].min(), first_visits["altitude"].max(), 25)
bottom = np.zeros(len(alt_bins) - 1)
for s in survey_order:
    subset = first_visits.loc[first_visits["survey"] == s, "altitude"]
    counts, _ = np.histogram(subset, bins=alt_bins)
    axes[1].bar(alt_bins[:-1], counts, width=np.diff(alt_bins), align="edge", bottom=bottom,
                color=survey_colors[s], edgecolor="black", linewidth=0.3, label=s, zorder=3)
    bottom += counts
axes[1].axvline(ALIGNMENT["el"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
                label=f"initial alignment (El={ALIGNMENT['el']})")
axes[1].set_xlabel("altitude (deg)")
axes[1].set_title("Starting altitude by survey, excl. ToO")
axes[1].legend(title="survey", loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, fontsize=9)

for ax in axes:
    style(ax)
plt.tight_layout()
plt.show()


## 9. Starting filter composition by survey, stacked, with the alignment filter marked

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bottom = np.zeros(len(band_order))
for s in survey_order:
    counts = (first_visits.loc[first_visits["survey"] == s, "band"]
              .value_counts().reindex(band_order).fillna(0).values)
    ax.bar(band_order, counts, bottom=bottom, color=survey_colors[s],
           edgecolor="black", linewidth=0.4, label=s, zorder=3)
    bottom += counts

align_idx = band_order.index(ALIGNMENT["filter"])
align_bar = bars[align_idx]
ax.annotate("Current IA",
            xy=(align_bar.get_x() + align_bar.get_width()/2, align_bar.get_height()*1.5),
            xytext=(0, 48), textcoords="offset points", ha="center", fontsize=9,
            color=ALIGNMENT_COLOR,
            arrowprops=dict(arrowstyle="->", color=ALIGNMENT_COLOR))

ax.set_xlabel("Filter (band)"); ax.set_ylabel("Number of nights")
ax.set_title("Starting filter composition by survey, excl. ToO")
ax.legend(title="survey", loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, fontsize=9)
style(ax)
plt.tight_layout()
plt.show()


***
Just Year 1
***

In [ ]:
first_visits = pd.read_csv("first_visit_per_night.csv")
first_visits = first_visits[first_visits["survey"] != "ToO"]

YEAR = 1
nights_per_year = 365.25
first_visits = first_visits[
    (first_visits["night"] > (YEAR - 1) * nights_per_year) &
    (first_visits["night"] <= YEAR * nights_per_year)
].reset_index(drop=True)

n_nights = len(first_visits)
print(f"{n_nights} nights in Year {YEAR}")

## Summary statistics

In [ ]:
summary_cols = ["azimuth", "altitude", "rotTelPos", "rotSkyPos"]
first_visits[summary_cols].describe()


In [ ]:
first_visits["band"].value_counts()


## Nights starting in each filter, with the alignment filter (i-band) marked

In [ ]:
band_counts = first_visits["band"].value_counts().reindex(band_order).fillna(0)

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(band_counts.index, band_counts.values,
              color=[filter_colors[b] for b in band_counts.index],
              edgecolor="black", linewidth=0.6, zorder=3)
for bar, cnt in zip(bars, band_counts.values):
    ax.annotate(f"{int(cnt)}", (bar.get_x() + bar.get_width()/2, bar.get_height()),
                textcoords="offset points", xytext=(0, 4), ha="center", fontsize=10)

# mark the current initial-alignment filter
align_idx = band_order.index(ALIGNMENT["filter"])
align_bar = bars[align_idx]
ax.annotate("Current IA",
            xy=(align_bar.get_x() + align_bar.get_width()/2, align_bar.get_height()*1.5),
            xytext=(0, 48), textcoords="offset points", ha="center", fontsize=9,
            color=ALIGNMENT_COLOR,
            arrowprops=dict(arrowstyle="->", color=ALIGNMENT_COLOR))

ax.set_ylabel("Number of nights")
ax.set_xlabel("Filter (band)")
ax.set_title(f"Nights starting in each filter, excl. ToO (n={n_nights})")
style(ax)
plt.tight_layout()
plt.show()


## Azimuth / altitude / rotator histograms, with alignment target marked

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(first_visits["azimuth"], bins=36, color="#3b6fa0", edgecolor="black", linewidth=0.5, zorder=3)
axes[0].axvline(ALIGNMENT["az"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
                label=f"initial alignment (Az={ALIGNMENT['az']})")
axes[0].set_title("Starting Azimuth"); axes[0].set_xlabel("azimuth (deg)"); axes[0].set_ylabel("Number of nights")
axes[0].legend(fontsize=8, frameon=False)

axes[1].hist(first_visits["altitude"], bins=36, color="#3b6fa0", edgecolor="black", linewidth=0.5, zorder=3)
axes[1].axvline(ALIGNMENT["el"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
                label=f"initial alignment (El={ALIGNMENT['el']})")
axes[1].set_title("Starting Altitude"); axes[1].set_xlabel("altitude (deg)")
axes[1].legend(fontsize=8, frameon=False)

axes[2].hist(first_visits["rotTelPos"], bins=36, color="#3b6fa0", edgecolor="black", linewidth=0.5, zorder=3)
axes[2].axvline(ALIGNMENT["rot"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
                label=f"initial alignment (Rot={ALIGNMENT['rot']})")
axes[2].set_title("Starting Rotator Angle (rotTelPos)"); axes[2].set_xlabel("rotTelPos (deg)")
axes[2].legend(fontsize=8, frameon=False)

for ax in axes:
    style(ax)
plt.suptitle(f"Distributions of first-visit-of-night pointing parameters, excl. ToO (n={n_nights})", y=1.03)
plt.tight_layout()
plt.show()


## Starting azimuth composition by filter, stacked

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(0, 360, 37)
bottom = np.zeros(len(bins) - 1)
for b in band_order:
    subset = first_visits.loc[first_visits["band"] == b, "azimuth"]
    counts, _ = np.histogram(subset, bins=bins)
    ax.bar(bins[:-1], counts, width=np.diff(bins), align="edge", bottom=bottom,
           color=filter_colors[b], edgecolor="black", linewidth=0.3, label=b, zorder=3)
    bottom += counts

ax.axvline(ALIGNMENT["az"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
           label=f"initial alignment (Az={ALIGNMENT['az']})")

ax.set_xlabel("azimuth (deg)"); ax.set_ylabel("Number of nights")
ax.set_title("Starting azimuth composition by filter, excl. ToO")
ax.legend(title="band", ncol=7, loc="upper center", bbox_to_anchor=(0.5, -0.15), frameon=False, fontsize=8)
style(ax)
plt.tight_layout()
plt.show()


## Starting azimuth vs. altitude, colored by survey, with alignment target marked

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for s in survey_order:
    sub = first_visits[first_visits["survey"] == s]
    if len(sub) == 0:
        continue
    ax.scatter(sub["azimuth"], sub["altitude"], s=18, alpha=0.7, color=survey_colors[s],
               edgecolor="none", label=f"{s} (n={len(sub)})", zorder=3)

ax.scatter([ALIGNMENT["az"]], [ALIGNMENT["el"]], marker="*", s=350, color="gold",
           edgecolor="black", linewidth=1.4, zorder=5,
           label=f"initial alignment (Az={ALIGNMENT['az']}, El={ALIGNMENT['el']})")

ax.set_xlabel("azimuth (deg)"); ax.set_ylabel("altitude (deg)"); ax.set_xlim(0, 360)
ax.set_title(f"Starting azimuth vs. altitude by survey, excl. ToO (n={n_nights})")
ax.legend(title="survey", loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, fontsize=9)
style(ax)
plt.tight_layout()
plt.show()


## Azimuth and altitude, stacked by survey, with alignment target marked

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

az_bins = np.linspace(0, 360, 37)
bottom = np.zeros(len(az_bins) - 1)
for s in survey_order:
    subset = first_visits.loc[first_visits["survey"] == s, "azimuth"]
    counts, _ = np.histogram(subset, bins=az_bins)
    axes[0].bar(az_bins[:-1], counts, width=np.diff(az_bins), align="edge", bottom=bottom,
                color=survey_colors[s], edgecolor="black", linewidth=0.3, label=s, zorder=3)
    bottom += counts
axes[0].axvline(ALIGNMENT["az"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
                label=f"initial alignment (Az={ALIGNMENT['az']})")
axes[0].set_xlabel("azimuth (deg)"); axes[0].set_ylabel("Number of nights")
axes[0].set_title("Starting azimuth by survey, excl. ToO")
axes[0].legend(fontsize=7, frameon=False, loc="upper right")

alt_bins = np.linspace(first_visits["altitude"].min(), first_visits["altitude"].max(), 25)
bottom = np.zeros(len(alt_bins) - 1)
for s in survey_order:
    subset = first_visits.loc[first_visits["survey"] == s, "altitude"]
    counts, _ = np.histogram(subset, bins=alt_bins)
    axes[1].bar(alt_bins[:-1], counts, width=np.diff(alt_bins), align="edge", bottom=bottom,
                color=survey_colors[s], edgecolor="black", linewidth=0.3, label=s, zorder=3)
    bottom += counts
axes[1].axvline(ALIGNMENT["el"], color=ALIGNMENT_COLOR, linestyle="--", linewidth=1.6, zorder=4,
                label=f"initial alignment (El={ALIGNMENT['el']})")
axes[1].set_xlabel("altitude (deg)")
axes[1].set_title("Starting altitude by survey, excl. ToO")
axes[1].legend(title="survey", loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, fontsize=9)

for ax in axes:
    style(ax)
plt.tight_layout()
plt.show()


## Starting filter composition by survey, stacked, with the alignment filter marked

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bottom = np.zeros(len(band_order))
for s in survey_order:
    counts = (first_visits.loc[first_visits["survey"] == s, "band"]
              .value_counts().reindex(band_order).fillna(0).values)
    ax.bar(band_order, counts, bottom=bottom, color=survey_colors[s],
           edgecolor="black", linewidth=0.4, label=s, zorder=3)
    bottom += counts

align_idx = band_order.index(ALIGNMENT["filter"])
align_bar = bars[align_idx]
ax.annotate("Current IA",
            xy=(align_bar.get_x() + align_bar.get_width()/2, align_bar.get_height()*1.5),
            xytext=(0, 48), textcoords="offset points", ha="center", fontsize=9,
            color=ALIGNMENT_COLOR,
            arrowprops=dict(arrowstyle="->", color=ALIGNMENT_COLOR))

ax.set_xlabel("Filter (band)"); ax.set_ylabel("Number of nights")
ax.set_title("Starting filter composition by survey, excl. ToO")
ax.legend(title="survey", loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, fontsize=9)
style(ax)
plt.tight_layout()
plt.show()
